# Train and Evaluate a Cross-Generator Run (VM workbook)

**Cross-Generator Generalization in Deepfake Detection (IE7374)**

Run this after `00_setup_and_preprocess.ipynb` has produced the crops cache
(`data/processed/` + `data/manifests/crops.parquet`) on this VM.

It builds the leave-one-manipulation-out splits, trains one detector on 3 of the 4
FF++ methods, tests it on the held-out 4th (plus the SimSwap set when it exists),
and prints the seen-vs-unseen gap.

**There are eight runs** (2 backbones x 4 held-out methods) and five of us, so we
split them up. Step 0 is the run board: it shows which runs are already done and
picks the next open one for you. **Claim your run in `RUNS.md` first** so two people
do not train the same one.

## 0. Run board -- pick your run
The first thing to run. Scans `configs/` for the eight runs and `experiments/results/`
for finished ones (a run is DONE once its results JSON has been pushed). It sets `RUN`
to the next unclaimed run for you; change it to whichever you claimed in `RUNS.md`.

**Do `git pull` before this** so the board reflects teammates' latest pushes.

In [ ]:
import glob, os

# work regardless of where Jupyter launched the kernel: hop to the repo root
# (the folder with configs/ and experiments/). handles kernels that start in
# the home dir or in experiments/ instead of the repo root.
if not os.path.isdir("configs"):
    for cand in ("..", os.path.expanduser("~/Cross-Generator-Generalization-in-Deepfake-Detection")):
        if os.path.isdir(os.path.join(cand, "configs")):
            os.chdir(cand)
            break
assert os.path.isdir("configs"), (
    f"can't find configs/ from {os.getcwd()}. cd to the repo root "
    "(the folder with configs/ and experiments/) and re-run this cell.")
print("repo root:", os.getcwd())

# the eight runs: 2 backbones (EfficientNet, XceptionNet) x 4 held-out FF++ methods,
# one committed config each in configs/
runs = sorted(os.path.splitext(os.path.basename(f))[0]
              for f in glob.glob("configs/*.yaml")
              if "preprocess" not in os.path.basename(f)
              and ".smoke" not in os.path.basename(f))

def is_done(r):  # done == its results JSON exists (whoever ran it pushed it)
    return os.path.exists(f"experiments/results/{r}.json")

todo = [r for r in runs if not is_done(r)]
print("=" * 62)
print("TRAINING RUN BOARD   (git pull first to see teammates' latest)")
print("=" * 62)
for r in runs:
    print(f"  [{'DONE' if is_done(r) else '    '}]  {r}")
print("-" * 62)
print(f"{len(runs) - len(todo)}/{len(runs)} done   |   {len(todo)} left")
if todo:
    print("suggested next (first open run):", todo[0])
print("=" * 62)

assert runs, "no run configs found in configs/ -- are you on an up-to-date main? (git pull)"

# ADJUST 1: the run you claimed in RUNS.md. Defaults to the first open run.
RUN = todo[0] if todo else runs[0]

# ADJUST 2 (optional, GPU hardware only): leave None to use the committed config.
AMP_OVERRIDE = None      # e.g. "fp16" on P100 / V100
BATCH_OVERRIDE = None    # e.g. 16 if VRAM is tight

assert RUN in runs, f"no config for {RUN}. Pick from: {runs}"
print("\nyou are running:", RUN)

## 1. Confirm the crops cache is present
If this fails, run `00_setup_and_preprocess.ipynb` first.

In [ ]:
import pandas as pd
assert os.path.exists("data/manifests/crops.parquet"), "run 00_setup_and_preprocess.ipynb first"
m = pd.read_parquet("data/manifests/crops.parquet")
print("crops:", len(m))
print(m.groupby("method").size())

## 2. Build the leave-one-manipulation-out splits
Writes one identity-disjoint fold per held-out method into `data/splits/`.
For each fold, train+val are 3 methods plus real, test is the held-out method plus real.

In [ ]:
!python data/make_splits.py --manifest data/manifests/crops.parquet --out data/splits
!ls -la data/splits

## 3. Load your run config
Loads the committed config for the `RUN` you picked in step 0 and validates its inputs.

The optional overrides from step 0 (`AMP_OVERRIDE` / `BATCH_OVERRIDE`) adapt to your GPU
without editing the committed config: when set, the notebook writes a gitignored
`configs/_local/` copy and trains from that, so the committed config stays authoritative
(seed, data, split, lr are never touched -- only how the run executes).

In [ ]:
import yaml

cfg_file = f"configs/{RUN}.yaml"
cfg = yaml.safe_load(open(cfg_file))
if AMP_OVERRIDE is not None:
    cfg["amp"] = AMP_OVERRIDE
if BATCH_OVERRIDE is not None:
    cfg["batch_size"] = BATCH_OVERRIDE
run_name = cfg["run_name"]

if AMP_OVERRIDE is not None or BATCH_OVERRIDE is not None:
    os.makedirs("configs/_local", exist_ok=True)
    cfg_path = f"configs/_local/{run_name}.yaml"
    yaml.safe_dump(cfg, open(cfg_path, "w"), sort_keys=False)
    print("using machine-local override:", cfg_path)
else:
    cfg_path = cfg_file
    print("using committed config:", cfg_path)

assert os.path.exists(cfg["manifest"]), "crops.parquet missing: run 00_setup_and_preprocess.ipynb"
assert os.path.exists(cfg["split"]), f"{cfg['split']} missing: run the splits cell (step 2)"
print(yaml.safe_dump(cfg, sort_keys=False))

## 4. Train
Fine-tunes the ImageNet-pretrained backbone on the 3 training methods plus real.
Saves a checkpoint under `checkpoints/<run_name>/`. Uses the GPU if one is visible
(check step 3 of the setup notebook). This is the long cell.

In [ ]:
!python experiments/train.py --config {cfg_path}

## 5. Evaluate
Scores the trained model on each method: the 3 seen methods (in-distribution) and
the held-out method (unseen), plus the SimSwap set if its split exists. Writes
`experiments/results/<run_name>.json` in the shared results schema.

In [ ]:
!python experiments/evaluate.py --config {cfg_path}

## 6. Read the result (the seen-vs-unseen gap)
The headline number is the drop from seen methods to the held-out (unseen) one.

In [ ]:
import json
r = json.load(open(f"experiments/results/{run_name}.json"))
df = pd.DataFrame(r["results"])
print(f"run: {r['run_name']}  backbone: {r['backbone']}  held-out: {r['held_out_method']}  level: {r['level']}")
print(df[["tested_on", "seen", "auc", "acc", "f1"]].to_string(index=False))
seen = df[df.seen == True]["auc"].mean()
unseen = df[df.seen == False]["auc"].mean()
print(f"\nmean seen AUC:   {seen:.4f}")
print(f"mean unseen AUC: {unseen:.4f}")
print(f"generalization gap (seen - unseen): {seen - unseen:.4f}")

## 7. Share your result with the team
Two small pushes: your results JSON (feeds the transfer matrix) and your row in
`RUNS.md` marked done. Checkpoints and crops are gitignored, so they never get pushed.
Without step 7 your run stays on your VM and the board still shows it TODO.

First time on a VM: make sure git is configured (`git config user.name` / `user.email`)
and that you can push (you are a repo collaborator; cloning over HTTPS caches your token).

In [ ]:
run_json = f"experiments/results/{run_name}.json"
assert os.path.exists(run_json), "no results JSON yet; run steps 4 and 5 first"
print("remember to mark", run_name, "done in RUNS.md, then:")
print("  git add", run_json, "RUNS.md")
print(f"  git commit -m 'results: {run_name}'")
print("  git pull --rebase && git push")
# uncomment to do it from here:
# !git add {run_json} RUNS.md
# !git commit -m "results: {run_name}"
# !git pull --rebase
# !git push

## Next steps
- Check the board in step 0: when all eight show DONE, assemble them with
  `02_transfer_matrix.ipynb`.
- Taking a second run? Change `RUN` in step 0 (claim it in `RUNS.md` first) and re-run.
- The held-out method is never opened during training, so its column is a clean
  unseen-generator measurement.
- The `SimSwap` column stays empty until the self-generated set and its split exist.